# CS2 EXP-3 — NeoBERT-250M Frozen Encoder Linear Probe

## 1. Dependencies

In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
    "einops",   # NeoBERT (chandar-lab/NeoBERT) remote modeling code dependency
], check=True)

# EXP-3 only needs the frozen NeoBERT encoder (no LoRA/PEFT), so unlike EXP-4
# this doesn't need `peft`/`accelerate`. It still needs `xformers`, pinned to
# match torch==2.5.1 for the same reason documented in EXP-4's install cell:
# an unpinned `-U xformers` silently upgrades torch out from under this pin.
# We force `use_unpadding=False` in case_study_2/models.py (see PDD sec. 5.2),
# so flash-attention is NOT required.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps",
    "xformers==0.0.28.post3",
], check=True)

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
print("Dependencies installed successfully for CUDA 12.1 driver!")


Dependencies installed successfully for CUDA 12.1 driver!


In [ ]:
import subprocess
nvidia_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(nvidia_smi.stdout or "nvidia-smi produced no stdout")
if nvidia_smi.stderr:
    print(nvidia_smi.stderr)

import os
print("CUDA_VISIBLE_DEVICES before override:", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))

import torch
print("torch.__version__:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())


In [3]:
import torch
try:
    torch.cuda.init()
except Exception as e:
    print(repr(e))


In [4]:
import subprocess, sys

# Fallback: only needed if the cell above shows torch.cuda.is_available()==False
# after the pinned install (seen occasionally on some CUDA 12.1 hosts). Re-run
# this, then RESTART THE KERNEL, then re-run from the top -- do not just
# continue in the same process, torch's CUDA init is one-shot per process.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu121",
], check=True)

print("Reinstalled. Restart the kernel now, then re-run your CUDA check cell.")


Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp310-cp310-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp310-cp310-linux_x86_64.whl (7.3 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.5 MB/s eta 0:00:00


Reinstalled. Restart the kernel now, then re-run your CUDA check cell.


In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"

DEVICE = "cuda:0"

print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")


Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 1.5 Settings

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "Recovery"

WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

DOWNSAMPLED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet"
MANIFEST_PATH = MANIFEST_ROOT / "cs1_shared_rotating_5fold_v1" / "project_grouped_5fold_manifest.parquet"

CODE_COLUMN = "normalized_code"
# CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

EXP3_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / f"exp3_neobert_linear_probe_v1_{CODE_COLUMN_TAG}"
EXP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_CACHE_DIR = DATA_ROOT / "embedding_cache" / "neobert_v1"
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"
# os.environ["HF_TOKEN"] = "secret"

RUN_SMOKE_TEST = True
RUN_PROFILE = True
RUN_OFFICIAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Downsampled parquet: {DOWNSAMPLED_PARQUET}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")


## 2. Clone the repository

In [7]:
import urllib.request
import zipfile
from pathlib import Path

if not REPO_ROOT.exists():
    print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")

    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = Path.cwd() / "repo_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(Path.cwd())

    repo_name = clean_url.split("/")[-1]
    extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

    zip_path.unlink()
    print("Repository setup complete!")
else:
    print(f"Repository already exists at {REPO_ROOT}")


Extracting files...
Repository setup complete!


## 3. Verify GPU, RAM, and storage budget

In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"
DEVICE = "cuda:0"
print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"bfloat16 supported: {torch.cuda.is_bf16_supported()}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Locked to single GPU: NVIDIA A100-SXM4-80GB
bfloat16 supported: True
Total VRAM: 84.99 GB


## 4. Data availability check

In [ ]:
STORAGE_CAP_GB = 60

required_data_files = {
    "downsampled parquet": DOWNSAMPLED_PARQUET,
    "shared 5-fold manifest": MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by notebooks/scope2_preprocessing.ipynb (downsampling + "
        "manifest-generation sections) and were previously synced through Google Drive. Copy "
        f"them into the paths above, or re-run that notebook. Keep an eye on the {STORAGE_CAP_GB} GB storage cap."
    )
    raise FileNotFoundError("Required processed data/manifest are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")


## 5. Write case_study_2 source files

Writes `models.py` (already NeoBERT-aware, unchanged from EXP-4), `data_loader.py`, and `exp3/exp3_linear_probe.py` into the cloned checkout.

In [10]:
(SRC_DIR / "case_study_2/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/__init__.py").write_text('')
print("Wrote", "case_study_2/__init__.py")


Wrote case_study_2/__init__.py


In [11]:
(SRC_DIR / "case_study_2/models.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/models.py").write_text('from __future__ import annotations\n\nimport os\nimport warnings\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\n\n\nDEFAULT_CODE_MODEL = "huggingface/CodeBERTa-small-v1"\nDEFAULT_CODE_TOKENIZER = "huggingface/CodeBERTa-small-v1"\n\n# NeoBERT-250M backbone (Chandar Research Lab); ships as trust_remote_code on the Hub.\nDEFAULT_NEOBERT_MODEL = "chandar-lab/NeoBERT"\nDEFAULT_NEOBERT_TOKENIZER = "chandar-lab/NeoBERT"\n\n# Substring match so NeoBERT forks/finetunes are still recognized.\n_NEOBERT_NAME_HINTS = ("neobert",)\n\n\ndef _is_neobert_model(model_name: str) -> bool:\n    """Check whether a model name refers to a NeoBERT-family checkpoint."""\n    name = (model_name or "").lower()\n    return any(hint in name for hint in _NEOBERT_NAME_HINTS)\n\n\ndef configure_huggingface_cache(hf_cache_dir: Optional[str] = None) -> None:\n    """Set the Hugging Face cache/env variables used across this project\'s downloads."""\n    if hf_cache_dir:\n        hf_cache_dir = str(hf_cache_dir)\n        os.environ.setdefault("HF_HOME", hf_cache_dir)\n        os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(Path(hf_cache_dir) / "hub"))\n    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")\n    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")\n    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")\n    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")\n    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\n\ndef _dtype_from_policy(dtype_policy: str, device: str) -> Optional[torch.dtype]:\n    """Resolve a torch dtype from a policy name and device."""\n    dtype_policy = (dtype_policy or "auto").lower()\n    device = str(device)\n    if dtype_policy == "float16":\n        return torch.float16 if device == "cuda" else torch.float32\n    if dtype_policy == "bfloat16":\n        return torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32\n    if dtype_policy == "float32":\n        return torch.float32\n    if dtype_policy == "auto":\n        if device == "cuda" and torch.cuda.is_bf16_supported():\n            return torch.bfloat16\n        if device == "cuda":\n            return torch.float32\n        return torch.float32\n    raise ValueError(f"Unknown dtype_policy: {dtype_policy}")\n\n\ndef _apply_neobert_config_overrides(config: Any) -> Any:\n    """Disable NeoBERT sequence unpadding, since we pad batches instead of packing them."""\n    candidate_flags = ("use_unpadding", "unpad_inputs", "unpad", "pack_sequences")\n    matched = False\n    for flag in candidate_flags:\n        if hasattr(config, flag):\n            setattr(config, flag, False)\n            matched = True\n    if not matched:\n        warnings.warn(\n            "[models] No known unpadding flag found on the NeoBERT config "\n            f"(checked: {candidate_flags}); verify attention-mask correctness manually."\n        )\n    return config\n\n\ndef load_code_tokenizer(\n    tokenizer_name: str = DEFAULT_CODE_TOKENIZER,\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n):\n    """Load the tokenizer for a code model, auto-detecting NeoBERT\'s trust_remote_code need."""\n    configure_huggingface_cache(hf_cache_dir)\n    if trust_remote_code is None:\n        trust_remote_code = _is_neobert_model(tokenizer_name)\n    return AutoTokenizer.from_pretrained(\n        tokenizer_name,\n        use_fast=True,\n        cache_dir=hf_cache_dir,\n        trust_remote_code=trust_remote_code,\n    )\n\n\ndef load_code_encoder(\n    model_name: str = DEFAULT_CODE_MODEL,\n    dtype_policy: str = "auto",\n    device: Optional[str] = None,\n    freeze: bool = True,\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n) -> nn.Module:\n    """Load a code backbone encoder, applying NeoBERT-specific safeguards when needed."""\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    configure_huggingface_cache(hf_cache_dir)\n    dtype = _dtype_from_policy(dtype_policy, device)\n\n    is_neobert = _is_neobert_model(model_name)\n    if trust_remote_code is None:\n        trust_remote_code = is_neobert\n\n    kwargs: Dict[str, Any] = {"cache_dir": hf_cache_dir, "trust_remote_code": trust_remote_code}\n    if dtype is not None:\n        kwargs["torch_dtype"] = dtype\n\n    if is_neobert:\n        # NeoBERT\'s fused flash/memory-efficient SDPA backends crash with a\n        # device-side CUDA assert on real (non-toy) batches on this\n        # environment; force the math (unfused) backend instead.\n        torch.backends.cuda.enable_flash_sdp(False)\n        torch.backends.cuda.enable_mem_efficient_sdp(False)\n        torch.backends.cuda.enable_math_sdp(True)\n\n        # Patch the config before the backbone is instantiated.\n        config = AutoConfig.from_pretrained(\n            model_name, cache_dir=hf_cache_dir, trust_remote_code=trust_remote_code\n        )\n        config = _apply_neobert_config_overrides(config)\n        kwargs["config"] = config\n\n    model = AutoModel.from_pretrained(model_name, **kwargs)\n    model.to(device)\n\n    if freeze:\n        for param in model.parameters():\n            param.requires_grad = False\n        model.eval()\n\n    return model\n\n\ndef mean_pool_last_hidden(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    """Mean-pool token embeddings over non-padded positions."""\n    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)\n    summed = (last_hidden_state * mask).sum(dim=1)\n    denom = mask.sum(dim=1).clamp(min=1.0)\n    return summed / denom\n\n\ndef cls_pool_last_hidden(last_hidden_state: torch.Tensor) -> torch.Tensor:\n    """Take the [CLS]-position embedding."""\n    return last_hidden_state[:, 0, :]\n\n\nclass CodeSequenceClassifier(nn.Module):\n    """Backbone encoder + linear classification head over pooled embeddings."""\n\n    def __init__(\n        self,\n        model_name: str = DEFAULT_CODE_MODEL,\n        num_labels: int = 1,\n        freeze_backbone: bool = False,\n        pooling: str = "mean",\n        dtype_policy: str = "auto",\n        hf_cache_dir: Optional[str] = None,\n        trust_remote_code: Optional[bool] = None,\n        enforce_fp32_head: Optional[bool] = None,\n    ) -> None:\n        """Build the backbone and classification head."""\n        super().__init__()\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        self.backbone = load_code_encoder(\n            model_name=model_name,\n            dtype_policy=dtype_policy,\n            device=device,\n            freeze=freeze_backbone,\n            hf_cache_dir=hf_cache_dir,\n            trust_remote_code=trust_remote_code,\n        )\n        hidden_size = int(self.backbone.config.hidden_size)\n        self.classification_head = nn.Linear(hidden_size, num_labels)\n        self.pooling = pooling\n\n        # Pooling and the classification head run in float32 regardless of\n        # ambient autocast dtype, to avoid baking a bf16/fp16 NaN/Inf from\n        # NeoBERT\'s attention stack into the trainable head.\n        if enforce_fp32_head is None:\n            enforce_fp32_head = _is_neobert_model(model_name)\n        self.enforce_fp32_head = enforce_fp32_head\n\n    @property\n    def config(self):\n        """Expose the underlying backbone config to peft."""\n        return self.backbone.config\n\n    @property\n    def device(self) -> torch.device:\n        """Expose the device where parameters reside."""\n        return next(self.parameters()).device\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, **kwargs: Any) -> torch.Tensor:\n        """Encode, pool, and classify a batch, returning logits."""\n        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)\n        hidden = outputs.last_hidden_state\n\n        if self.enforce_fp32_head:\n            hidden = hidden.float()\n            attention_mask_for_pool = attention_mask.float()\n        else:\n            attention_mask_for_pool = attention_mask\n\n        if self.pooling == "cls":\n            pooled = cls_pool_last_hidden(hidden)\n        else:\n            pooled = mean_pool_last_hidden(hidden, attention_mask_for_pool)\n\n        if self.enforce_fp32_head:\n            # Disable autocast so the head matmul isn\'t downcast back to bf16/fp16.\n            with torch.autocast(device_type=pooled.device.type, enabled=False):\n                logits = self.classification_head(pooled.float())\n        else:\n            logits = self.classification_head(pooled)\n\n        if logits.ndim > 1 and logits.size(-1) == 1:\n            return logits.squeeze(-1)\n        return logits\n\n\ndef count_trainable_parameters(model: nn.Module) -> Dict[str, int]:\n    """Report trainable vs. total parameter counts."""\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total = sum(p.numel() for p in model.parameters())\n    return {\n        "trainable_parameters": int(trainable),\n        "total_parameters": int(total),\n        "trainable_percent": float(100.0 * trainable / max(total, 1)),\n    }\n\n\ndef infer_lora_target_modules(model: nn.Module) -> List[str]:\n    """Guess which attention projection module names LoRA should target."""\n    module_names = [name for name, _ in model.named_modules()]\n    candidate_sets = [\n        ["qkv"],\n        ["q_proj", "v_proj"],\n        ["query", "value"],\n        ["in_proj"],\n    ]\n    for candidates in candidate_sets:\n        if all(any(name.endswith(candidate) or f".{candidate}" in name for name in module_names) for candidate in candidates):\n            return candidates\n    return ["query", "value"]\n\n\ndef create_lora_sequence_classifier(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    lora_dropout: float = 0.05,\n    pooling: str = "mean",\n    dtype_policy: str = "auto",\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n):\n    """Wrap a CodeSequenceClassifier with a LoRA adapter via peft."""\n    from peft import LoraConfig, get_peft_model\n\n    base = CodeSequenceClassifier(\n        model_name=model_name,\n        freeze_backbone=False,\n        pooling=pooling,\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n        trust_remote_code=trust_remote_code,\n    )\n    target_modules = infer_lora_target_modules(base)\n    config = LoraConfig(\n        r=rank,\n        lora_alpha=lora_alpha,\n        target_modules=target_modules,\n        lora_dropout=lora_dropout,\n        bias="none",\n        task_type="FEATURE_EXTRACTION",\n        modules_to_save=["classification_head"],\n    )\n    return get_peft_model(base, config)\n\n\ndef get_lora_model(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    pooling: str = "mean",\n    trust_remote_code: Optional[bool] = None,\n):\n    """Build a LoRA-adapted sequence classifier for the given backbone."""\n    return create_lora_sequence_classifier(\n        model_name=model_name,\n        rank=rank,\n        lora_alpha=lora_alpha,\n        pooling=pooling,\n        trust_remote_code=trust_remote_code,\n    )\n\n')
print("Wrote", "case_study_2/models.py")


Wrote case_study_2/models.py


In [12]:
(SRC_DIR / "case_study_2/data_loader.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/data_loader.py").write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional, Dict, Any, List\n\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\n\n\nEMPTY_CODE_SENTINEL = "EMPTY_CODE_SAMPLE"\n\n\nclass CodeTextDataset(Dataset):\n    """PyTorch Dataset yielding raw code text plus label/id/project for one row."""\n\n    def __init__(\n        self,\n        dataframe: pd.DataFrame,\n        code_column: str = "normalized_code",\n        label_column: str = "label",\n        source_id_column: str = "source_row_id",\n        project_column: str = "project",\n    ) -> None:\n        """Copy the frame and replace empty code with a sentinel token."""\n        self.df = dataframe.copy().reset_index(drop=True)\n        self.code_column = code_column\n        self.label_column = label_column\n        self.source_id_column = source_id_column\n        self.project_column = project_column\n\n        self.df[self.code_column] = self.df[self.code_column].fillna("").astype(str)\n        empty_mask = self.df[self.code_column].str.strip().eq("")\n        if empty_mask.any():\n            self.df.loc[empty_mask, self.code_column] = EMPTY_CODE_SENTINEL\n\n    def __len__(self) -> int:\n        """Return the number of rows."""\n        return int(len(self.df))\n\n    def __getitem__(self, idx: int) -> Dict[str, Any]:\n        """Return one row as a plain dict."""\n        row = self.df.iloc[idx]\n        return {\n            "code": str(row[self.code_column]),\n            "label": int(row[self.label_column]),\n            "source_row_id": int(row[self.source_id_column]),\n            "project": str(row[self.project_column]),\n        }\n\n\n@dataclass\nclass TransformerBatchCollator:\n    """Tokenize a batch of raw-code dicts into padded model inputs."""\n\n    tokenizer: Any\n    max_length: int = 512\n    pad_to_multiple_of: Optional[int] = 8\n\n    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:\n        """Tokenize and collate a list of row dicts into one batch."""\n        texts = [feature["code"] for feature in features]\n        enc = self.tokenizer(\n            texts,\n            truncation=True,\n            max_length=self.max_length,\n            padding=True,\n            pad_to_multiple_of=self.pad_to_multiple_of,\n            return_tensors="pt",\n        )\n\n        labels = torch.tensor([feature["label"] for feature in features], dtype=torch.float32)\n        source_row_ids = torch.tensor([feature["source_row_id"] for feature in features], dtype=torch.long)\n        projects = [feature["project"] for feature in features]\n\n        enc["labels"] = labels\n        enc["label"] = labels\n        enc["source_row_id"] = source_row_ids\n        enc["project"] = projects\n        return enc\n\n\ndef create_dataloader(\n    dataframe: pd.DataFrame,\n    tokenizer: Any,\n    batch_size: int = 16,\n    max_length: int = 512,\n    shuffle: bool = False,\n    code_column: str = "normalized_code",\n    label_column: str = "label",\n    source_id_column: str = "source_row_id",\n    project_column: str = "project",\n    num_workers: int = 0,\n) -> DataLoader:\n    """Build a DataLoader that tokenizes code rows on the fly."""\n    dataset = CodeTextDataset(\n        dataframe=dataframe,\n        code_column=code_column,\n        label_column=label_column,\n        source_id_column=source_id_column,\n        project_column=project_column,\n    )\n    collator = TransformerBatchCollator(\n        tokenizer=tokenizer,\n        max_length=max_length,\n        pad_to_multiple_of=8 if torch.cuda.is_available() else None,\n    )\n    return DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        drop_last=False,\n        num_workers=num_workers,\n        pin_memory=torch.cuda.is_available(),\n        collate_fn=collator,\n    )\n\n\ndef get_pos_weight(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    """Compute the negative/positive ratio for BCEWithLogitsLoss\'s pos_weight."""\n    y = dataframe[label_column].astype(int).values\n    neg = int((y == 0).sum())\n    pos = int((y == 1).sum())\n    if pos == 0:\n        return torch.tensor([1.0], dtype=torch.float32)\n    return torch.tensor([neg / pos], dtype=torch.float32)\n\n\ndef get_class_weights(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    """Alias for get_pos_weight, used as the BCE positive-class weight."""\n    return get_pos_weight(dataframe, label_column=label_column)\n\n')
print("Wrote", "case_study_2/data_loader.py")


Wrote case_study_2/data_loader.py


In [13]:
(SRC_DIR / "case_study_2/exp3/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp3/__init__.py").write_text('')
print("Wrote", "case_study_2/exp3/__init__.py")


Wrote case_study_2/exp6/__init__.py


In [14]:
(SRC_DIR / "case_study_2/exp3/exp3_linear_probe.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp3/exp3_linear_probe.py").write_text('from __future__ import annotations\n\nimport gc\nimport json\nimport time\nfrom dataclasses import dataclass, asdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score\nfrom sklearn.preprocessing import StandardScaler\n\nfrom case_study_2.data_loader import create_dataloader\nfrom case_study_2.models import DEFAULT_NEOBERT_MODEL, DEFAULT_NEOBERT_TOKENIZER\nfrom utils import evaluation\nfrom utils import split_manifest\nfrom utils.evaluation import EvaluationConfig, select_f1_threshold\n\n\nEXP3_VERSION = "cs2-exp3-neobert-linear-probe-v2-nested-rotating-5fold"\n\n\n@dataclass(frozen=True)\nclass Exp3Config:\n    """Declared reproducible configuration for CS2-EXP3 (frozen NeoBERT embeddings + logistic probe)."""\n\n    experiment_name: str = "cs2_exp3_neobert_linear_probe"\n\n    code_column: str = "normalized_code"\n    source_id_column: str = "source_row_id"\n    label_column: str = "label"\n    project_column: str = "project"\n    fold_column: str = "fold"\n\n    model_name: str = DEFAULT_NEOBERT_MODEL\n    tokenizer_name: str = DEFAULT_NEOBERT_TOKENIZER\n    hf_cache_dir: Optional[str] = None\n    # NeoBERT supports up to 4096 tokens; kept at 512 since most functions fit.\n    max_length: int = 512\n    dtype_policy: str = "bfloat16"\n    embedding_batch_size: int = 64\n    pooling: str = "mean"\n    trust_remote_code: bool = True  # NeoBERT ships as trust_remote_code on the Hub\n\n    logistic_max_iter: int = 2000\n    logistic_solver: str = "lbfgs"\n    class_weight: str = "balanced"\n\n    n_splits: int = 5\n    random_state: int = 42\n    verbose: bool = True\n\n\n@dataclass(frozen=True)\nclass NestedProbeConfig:\n    """Inner-CV search configuration: C and decision threshold, selected per outer fold."""\n\n    C_grid: Tuple[float, ...] = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0)\n    inner_n_splits: int = 3\n    inner_random_state: int = 20260707\n    selection_metric: str = "average_precision_pr_auc"\n    verbose: bool = True\n\n\ndef resolve_device(require_cuda: bool = True) -> str:\n    """Pick the GPU when available; raise if a GPU is required but absent."""\n    if torch.cuda.is_available():\n        return "cuda:0"\n    if require_cuda:\n        raise RuntimeError("EXP-3 linear probe requires a CUDA GPU for embedding extraction.")\n    return "cpu"\n\n\n@torch.no_grad()\ndef extract_embeddings(\n    encoder,\n    tokenizer,\n    frame: pd.DataFrame,\n    config: Exp3Config,\n    device: str,\n    cache_path: Optional[Path] = None,\n    checkpoint_every: int = 100,\n) -> np.ndarray:\n    """Extract frozen mean/CLS-pooled NeoBERT embeddings for every row, once for the whole dataset."""\n    if cache_path is not None and cache_path.exists():\n        return np.load(cache_path)\n\n    if not str(device).startswith("cuda"):\n        raise RuntimeError("extract_embeddings must run on a CUDA device.")\n\n    frame = frame.reset_index(drop=True)\n    code_lengths = frame[config.code_column].fillna("").astype(str).str.len()\n    sort_order = code_lengths.sort_values(kind="mergesort").index.to_numpy()\n    sorted_frame = frame.iloc[sort_order].reset_index(drop=True)\n    inverse_order = np.argsort(sort_order)\n\n    n_rows = len(sorted_frame)\n    n_batches_total = -(-n_rows // config.embedding_batch_size)\n\n    checkpoint_path = cache_path.with_suffix(".checkpoint.npz") if cache_path is not None else None\n    all_embeddings: List[np.ndarray] = []\n    start_batch = 0\n\n    if checkpoint_path is not None and checkpoint_path.exists():\n        ckpt = np.load(checkpoint_path)\n        all_embeddings = [ckpt["embeddings"]]\n        start_batch = int(ckpt["n_batches"])\n\n    loader = create_dataloader(\n        sorted_frame,\n        tokenizer,\n        batch_size=config.embedding_batch_size,\n        max_length=config.max_length,\n        shuffle=False,\n        code_column=config.code_column,\n        label_column=config.label_column,\n        source_id_column=config.source_id_column,\n        project_column=config.project_column,\n        num_workers=2,\n    )\n\n    encoder.eval()\n    t0 = time.time()\n\n    for i, batch in enumerate(loader):\n        if i < start_batch:\n            continue\n\n        input_ids = batch["input_ids"].to(device, non_blocking=True)\n        attention_mask = batch["attention_mask"].to(device, non_blocking=True)\n\n        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n            outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)\n            # Pooling in fp32 guards against the known NeoBERT bf16 NaN/Inf\n            # instability (chandar-lab/NeoBERT GitHub Issue #11).\n            last_hidden = outputs.last_hidden_state.float()\n            if config.pooling == "cls":\n                pooled = last_hidden[:, 0, :]\n            else:\n                mask = attention_mask.unsqueeze(-1).to(last_hidden.dtype)\n                summed = (last_hidden * mask).sum(dim=1)\n                denom = mask.sum(dim=1).clamp(min=1.0)\n                pooled = summed / denom\n\n        if not torch.isfinite(pooled).all():\n            n_bad = (~torch.isfinite(pooled)).any(dim=-1).sum().item()\n            raise RuntimeError(\n                f"[embed] NaN/Inf detected in pooled embeddings for {n_bad}/{pooled.shape[0]} "\n                f"rows in batch {i} (known NeoBERT numerical instability, not a data problem)."\n            )\n\n        all_embeddings.append(pooled.detach().cpu().numpy())\n\n        if checkpoint_path is not None and (i + 1) % checkpoint_every == 0:\n            partial = np.concatenate(all_embeddings, axis=0)\n            np.savez(checkpoint_path, embeddings=partial, n_batches=i + 1)\n            if config.verbose:\n                elapsed_min = (time.time() - t0) / 60\n                print(f"  [embed] checkpoint @ batch {i+1}/{n_batches_total} | elapsed {elapsed_min:.1f} min")\n\n    embeddings_sorted = np.concatenate(all_embeddings, axis=0)\n    embeddings = embeddings_sorted[inverse_order]\n\n    if cache_path is not None:\n        cache_path.parent.mkdir(parents=True, exist_ok=True)\n        np.save(cache_path, embeddings)\n        if checkpoint_path is not None and checkpoint_path.exists():\n            checkpoint_path.unlink()\n\n    return embeddings\n\n\ndef _fit_probe(X: np.ndarray, y: np.ndarray, C: float, config: Exp3Config) -> Tuple[StandardScaler, LogisticRegression]:\n    """Fit a StandardScaler + L2 logistic regression probe at one C value."""\n    scaler = StandardScaler(copy=False)\n    X_s = scaler.fit_transform(X.astype(np.float32, copy=False))\n    clf = LogisticRegression(\n        C=C, max_iter=config.logistic_max_iter, solver=config.logistic_solver,\n        class_weight=config.class_weight, random_state=config.random_state,\n    )\n    clf.fit(X_s, y)\n    return scaler, clf\n\n\ndef _predict_probe(scaler: StandardScaler, clf: LogisticRegression, X: np.ndarray) -> np.ndarray:\n    """Score embeddings with an already-fit scaler + probe."""\n    return clf.predict_proba(scaler.transform(X.astype(np.float32, copy=False)))[:, 1]\n\n\ndef _inner_manifest(frame: pd.DataFrame, config: Exp3Config, nested_config: NestedProbeConfig, outer_fold_id: int) -> pd.DataFrame:\n    """Project-grouped, stratified inner-fold assignment for one outer fold\'s training rows."""\n    inner_split_config = split_manifest.SplitConfig(\n        n_splits=nested_config.inner_n_splits,\n        random_state=nested_config.inner_random_state + int(outer_fold_id),\n        source_id_column=config.source_id_column,\n        label_column=config.label_column,\n        group_column=config.project_column,\n    )\n    return split_manifest.create_project_grouped_manifest(\n        frame[[config.source_id_column, config.label_column, config.project_column]], config=inner_split_config\n    )\n\n\ndef run_exp3_nested_inner_profile(\n    dataset_frame: pd.DataFrame,\n    dataset_embeddings: np.ndarray,\n    manifest: pd.DataFrame,\n    outer_fold_id: int,\n    base_config: Exp3Config,\n    nested_config: NestedProbeConfig,\n) -> Dict[str, Any]:\n    """Select C and the decision threshold via 3-fold inner CV (micro-averaged/pooled PR-AUC) on this fold\'s training rows."""\n    t0 = time.time()\n    id_to_pos = {rid: pos for pos, rid in enumerate(dataset_frame[base_config.source_id_column].values)}\n\n    outer_train_ids = set(\n        manifest.loc[manifest[base_config.fold_column] != outer_fold_id, base_config.source_id_column]\n    )\n    train_frame = dataset_frame[dataset_frame[base_config.source_id_column].isin(outer_train_ids)].reset_index(drop=True)\n\n    inner_manifest_df = _inner_manifest(train_frame, base_config, nested_config, outer_fold_id)\n\n    rows = []\n    predictions_by_C: Dict[float, List[pd.DataFrame]] = {C: [] for C in nested_config.C_grid}\n    for inner_id in range(nested_config.inner_n_splits):\n        tr_ids = set(inner_manifest_df.loc[inner_manifest_df["fold"] != inner_id, "source_row_id"])\n        va_ids = set(inner_manifest_df.loc[inner_manifest_df["fold"] == inner_id, "source_row_id"])\n        tr_frame = train_frame[train_frame[base_config.source_id_column].isin(tr_ids)]\n        va_frame = train_frame[train_frame[base_config.source_id_column].isin(va_ids)]\n        tr_pos = [id_to_pos[rid] for rid in tr_frame[base_config.source_id_column].values]\n        va_pos = [id_to_pos[rid] for rid in va_frame[base_config.source_id_column].values]\n\n        X_tr = dataset_embeddings[tr_pos]\n        y_tr = tr_frame[base_config.label_column].astype(int).values\n        X_va = dataset_embeddings[va_pos]\n        y_va = va_frame[base_config.label_column].astype(int).values\n\n        for C in nested_config.C_grid:\n            scaler, clf = _fit_probe(X_tr, y_tr, C, base_config)\n            scores = _predict_probe(scaler, clf, X_va)\n            predictions_by_C[C].append(\n                pd.DataFrame({"source_row_id": va_frame[base_config.source_id_column].values, "label": y_va, "y_score": scores})\n            )\n            del scaler, clf, scores\n            gc.collect()\n\n        del tr_frame, va_frame, X_tr, X_va, y_tr, y_va\n        gc.collect()\n\n    # Micro-averaged (pooled) selection: concatenate every inner fold\'s held-out\n    # predictions for a given C and score once, instead of averaging separately\n    # computed per-fold scores (macro-average). PR-AUC/ROC-AUC are rank-based and\n    # not additive across folds, so macro-averaging them is a biased/noisy proxy\n    # when positives are rare; pooling first is the standard fix (as in sklearn\'s\n    # own average_precision_score computed over a full held-out set).\n    for C in nested_config.C_grid:\n        pooled = pd.concat(predictions_by_C[C], ignore_index=True)\n        pooled_pr_auc = float(average_precision_score(pooled["label"], pooled["y_score"]))\n        rows.append({"C": C, "inner_pooled_pr_auc": pooled_pr_auc, "n_val": len(pooled)})\n\n    C_summary = (\n        pd.DataFrame(rows)\n        .sort_values(["inner_pooled_pr_auc", "C"], ascending=[False, True], kind="stable")\n        .reset_index(drop=True)\n    )\n    selected_C = float(C_summary.iloc[0]["C"])\n    pooled_predictions = pd.concat(predictions_by_C[selected_C], ignore_index=True)\n    selected_threshold, threshold_metrics = select_f1_threshold(pooled_predictions["label"], pooled_predictions["y_score"])\n\n    del train_frame\n    gc.collect()\n\n    return {\n        "outer_fold_id": outer_fold_id,\n        "selected": {\n            "outer_fold_id": outer_fold_id,\n            "selected_C": selected_C,\n            "decision_threshold": selected_threshold,\n            "inner_pooled_pr_auc": float(C_summary.iloc[0]["inner_pooled_pr_auc"]),\n            "inner_validation_f1": float(threshold_metrics["f1"]),\n        },\n        "C_summary": C_summary,\n        "total_profile_seconds": time.time() - t0,\n    }\n\n\ndef _checkpoint_paths(output_dir: Path, outer_fold_id: int) -> Dict[str, Path]:\n    """Filesystem locations for one outer fold\'s resumable checkpoint."""\n    root = output_dir / "checkpoints"\n    root.mkdir(parents=True, exist_ok=True)\n    prefix = f"outer_fold_{outer_fold_id}"\n    return {\n        "predictions": root / f"{prefix}_predictions.parquet",\n        "selected": root / f"{prefix}_selected.json",\n        "training": root / f"{prefix}_outer_training.json",\n    }\n\n\ndef _write_outer_checkpoint(output_dir: Path, outer_fold_id: int, predictions: pd.DataFrame, selected: dict, training: dict) -> None:\n    """Persist one outer fold\'s result so a later run can resume without refitting."""\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    predictions.to_parquet(paths["predictions"], index=False)\n    with paths["selected"].open("w", encoding="utf-8") as f:\n        json.dump(selected, f, indent=2, default=str)\n    with paths["training"].open("w", encoding="utf-8") as f:\n        json.dump(training, f, indent=2, default=str)\n\n\ndef _load_outer_checkpoint(output_dir: Path, outer_fold_id: int) -> Optional[dict]:\n    """Load one outer fold\'s checkpoint if it exists and is complete."""\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    if not all(p.exists() for p in paths.values()):\n        return None\n    with paths["selected"].open("r", encoding="utf-8") as f:\n        selected = json.load(f)\n    with paths["training"].open("r", encoding="utf-8") as f:\n        training = json.load(f)\n    return {"predictions": pd.read_parquet(paths["predictions"]), "selected": selected, "training": training}\n\n\ndef _update_run_state(state_path: Path, completed_folds, status: str) -> None:\n    """Persist which outer folds are done, for resumability and progress inspection."""\n    state = {\n        "status": status,\n        "updated_utc": datetime.now(timezone.utc).isoformat(),\n        "completed_outer_folds": sorted(int(f) for f in completed_folds),\n    }\n    with state_path.open("w", encoding="utf-8") as f:\n        json.dump(state, f, indent=2)\n\n\ndef run_exp3_nested_probe(\n    dataset_frame: pd.DataFrame,\n    dataset_embeddings: np.ndarray,\n    manifest: pd.DataFrame,\n    base_config: Exp3Config,\n    nested_config: NestedProbeConfig,\n    output_dir: Path,\n    additional_metadata: Optional[Dict[str, Any]] = None,\n    resume: bool = True,\n) -> Dict[str, Any]:\n    """Run the official rotating 5-fold CS2-EXP3 experiment, with a 3-fold inner-CV search per outer fold."""\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    state_path = output_dir / "exp3_nested_run_state.json"\n    t0 = time.time()\n\n    id_to_pos = {rid: pos for pos, rid in enumerate(dataset_frame[base_config.source_id_column].values)}\n    fold_ids = sorted(manifest[base_config.fold_column].unique().tolist())\n\n    oof_parts = []\n    selected_rows = []\n    outer_training_rows = []\n    completed_folds = []\n\n    for outer_fold_id in fold_ids:\n        checkpoint = _load_outer_checkpoint(output_dir, outer_fold_id) if resume else None\n        if checkpoint is not None:\n            oof_parts.append(checkpoint["predictions"])\n            selected_rows.append(checkpoint["selected"])\n            outer_training_rows.append(checkpoint["training"])\n            completed_folds.append(outer_fold_id)\n            if nested_config.verbose:\n                print(f"  [nested] Outer fold {outer_fold_id}: loaded from checkpoint.")\n            continue\n\n        if nested_config.verbose:\n            print(f"  [nested] Outer fold {outer_fold_id}: inner C + threshold search...")\n\n        profile = run_exp3_nested_inner_profile(dataset_frame, dataset_embeddings, manifest, outer_fold_id, base_config, nested_config)\n        selected = profile["selected"]\n\n        train_ids = set(manifest.loc[manifest[base_config.fold_column] != outer_fold_id, base_config.source_id_column])\n        val_ids = set(manifest.loc[manifest[base_config.fold_column] == outer_fold_id, base_config.source_id_column])\n        if train_ids.intersection(val_ids):\n            raise RuntimeError(f"Outer fold {outer_fold_id}: train/val ID leakage detected.")\n\n        train_frame = dataset_frame[dataset_frame[base_config.source_id_column].isin(train_ids)]\n        val_frame = dataset_frame[dataset_frame[base_config.source_id_column].isin(val_ids)]\n\n        tr_pos = [id_to_pos[rid] for rid in train_frame[base_config.source_id_column].values]\n        va_pos = [id_to_pos[rid] for rid in val_frame[base_config.source_id_column].values]\n\n        X_tr = dataset_embeddings[tr_pos]\n        y_tr = train_frame[base_config.label_column].astype(int).values\n        X_va = dataset_embeddings[va_pos]\n\n        scaler, clf = _fit_probe(X_tr, y_tr, selected["selected_C"], base_config)\n        val_scores = _predict_probe(scaler, clf, X_va)\n\n        fold_oof = pd.DataFrame({\n            base_config.source_id_column: val_frame[base_config.source_id_column].values,\n            base_config.project_column: val_frame[base_config.project_column].values,\n            "label": val_frame[base_config.label_column].astype(int).values,\n            "y_score": val_scores,\n            "fold": outer_fold_id,\n        })\n\n        training_row = {\n            **selected,\n            "n_train": int(len(train_frame)),\n            "n_val": int(len(val_frame)),\n            "train_projects": int(train_frame[base_config.project_column].nunique()),\n            "val_projects": int(val_frame[base_config.project_column].nunique()),\n        }\n\n        _write_outer_checkpoint(output_dir, outer_fold_id, fold_oof, selected, training_row)\n\n        oof_parts.append(fold_oof)\n        selected_rows.append(selected)\n        outer_training_rows.append(training_row)\n        completed_folds.append(outer_fold_id)\n\n        _update_run_state(state_path, completed_folds, status="running")\n\n        del train_frame, val_frame, X_tr, X_va, y_tr, scaler, clf, val_scores, profile\n        gc.collect()\n\n    oof_predictions = pd.concat(oof_parts, axis=0).reset_index(drop=True)\n    selected_df = pd.DataFrame(selected_rows)\n    outer_training_df = pd.DataFrame(outer_training_rows)\n\n    mean_threshold = float(selected_df["decision_threshold"].mean())\n    eval_config = EvaluationConfig(threshold=mean_threshold, expected_n_folds=len(fold_ids))\n    eval_results = evaluation.evaluate_oof_predictions(oof_predictions, config=eval_config)\n\n    artifacts = {\n        "oof_predictions": output_dir / "exp3_nested_oof_predictions.parquet",\n        "selected_per_fold": output_dir / "exp3_selected_per_outer_fold.csv",\n        "outer_training_audit": output_dir / "exp3_outer_training_audit.csv",\n        "run_metadata": output_dir / "exp3_nested_run_metadata.json",\n    }\n    eval_results["predictions"].to_parquet(artifacts["oof_predictions"], index=False)\n    selected_df.to_csv(artifacts["selected_per_fold"], index=False)\n    outer_training_df.to_csv(artifacts["outer_training_audit"], index=False)\n\n    metadata = {\n        "exp3_version": EXP3_VERSION,\n        "base_config": asdict(base_config),\n        "nested_config": {**asdict(nested_config), "C_grid": list(nested_config.C_grid)},\n        "runtime_seconds": time.time() - t0,\n        **(additional_metadata or {}),\n    }\n    with open(artifacts["run_metadata"], "w", encoding="utf-8") as f:\n        json.dump(metadata, f, indent=2, default=str)\n\n    _update_run_state(state_path, completed_folds, status="completed")\n\n    return {\n        "oof_predictions": eval_results["predictions"],\n        "evaluation": eval_results,\n        "selected": selected_df,\n        "outer_fold_training": outer_training_df,\n        "artifacts": artifacts,\n    }\n\n\n__all__ = [\n    "EXP3_VERSION",\n    "Exp3Config",\n    "NestedProbeConfig",\n    "extract_embeddings",\n    "resolve_device",\n    "run_exp3_nested_inner_profile",\n    "run_exp3_nested_probe",\n]\n')
print("Wrote", "case_study_2/exp3/exp3_linear_probe.py")


Wrote case_study_2/exp6/exp6_linear_probe.py


## 6. Import project modules

In [ ]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1") or mod_name.startswith("utils"):
        del sys.modules[mod_name]

from case_study_2.data_loader import create_dataloader
from case_study_2.models import (
    configure_huggingface_cache, load_code_tokenizer, load_code_encoder,
    DEFAULT_NEOBERT_MODEL, DEFAULT_NEOBERT_TOKENIZER,
)
from case_study_2.exp3.exp3_linear_probe import (
    Exp3Config, NestedProbeConfig, extract_embeddings,
    run_exp3_nested_inner_profile, run_exp3_nested_probe,
)
from utils import split_manifest
from utils import evaluation
from utils.confidence_intervals import bootstrap_metric_ci, format_ci_report

print("Imported. Model:", DEFAULT_NEOBERT_MODEL)


## 7. Load the downsampled dataset and shared 5-fold manifest

In [ ]:
import pandas as pd

if not DOWNSAMPLED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing downsampled parquet: {DOWNSAMPLED_PARQUET}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_PATH}")

full_df = pd.read_parquet(DOWNSAMPLED_PARQUET)
manifest_df = split_manifest.load_manifest(MANIFEST_PATH, config=split_manifest.SplitConfig(n_splits=5, random_state=42))

print("full_df rows:", len(full_df))
print("manifest_df rows:", len(manifest_df))

fold_summary_diag = split_manifest.summarize_manifest(
    manifest_df, config=split_manifest.SplitConfig(n_splits=5, random_state=42)
)
display(fold_summary_diag)

fold_size_ratio = fold_summary_diag["test_rows"].max() / fold_summary_diag["test_rows"].min()
print(f"Fold test-size balance: smallest={fold_summary_diag['test_rows'].min()} rows, "
      f"largest={fold_summary_diag['test_rows'].max()} rows, ratio={fold_size_ratio:.2f}x")
if fold_size_ratio > 2.0:
    print("WARNING: fold sizes are notably imbalanced (ratio > 2x) -- "
          "regenerate the manifest via scope2_preprocessing.ipynb with an updated "
          "DOWNSAMPLE_MAX_ROWS_PER_PROJECT.")
else:
    print("Fold sizes look reasonably balanced.")


## 8. Build the dataset frame

In [ ]:
required_columns = {"source_row_id", "normalized_code", "abstracted_code_v1", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
manifest_ids = set(manifest_df["source_row_id"].tolist())
if manifest_ids != set(full_indexed.index):
    raise RuntimeError(
        "Manifest coverage does not match the downsampled dataset exactly. "
        f"Missing={len(set(full_indexed.index) - manifest_ids)}, extra={len(manifest_ids - set(full_indexed.index))}"
    )

dataset_frame = full_indexed.loc[list(manifest_ids)].reset_index(drop=True)
print("dataset_frame rows:", len(dataset_frame))
print("Unique projects:", dataset_frame["project"].nunique())


## 9. Configuration objects

In [ ]:
base_config = Exp3Config(hf_cache_dir=HF_CACHE_DIR, code_column=CODE_COLUMN)
nested_config = NestedProbeConfig()

print("Code column:", base_config.code_column)
print("C_grid:", nested_config.C_grid)
print("inner_n_splits:", nested_config.inner_n_splits)


## 10. Smoke test on a small subsample

NeoBERT ships as `trust_remote_code` and has a known numerical-instability bug (PDD sec. 5.2 / GitHub Issue #11). Cheap sanity check on ~300 rows -- tokenizer, encoder loading, embedding extraction (with the NaN guardrail), and a quick probe fit -- before committing to the full development+holdout embedding extraction.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_df = dataset_frame.sample(n=min(300, len(dataset_frame)), random_state=42).reset_index(drop=True)

    print("Loading tokenizer...")
    _smoke_tokenizer = load_code_tokenizer(base_config.tokenizer_name, hf_cache_dir=HF_CACHE_DIR)
    print("Loading encoder...")
    _smoke_encoder = load_code_encoder(
        base_config.model_name, dtype_policy=base_config.dtype_policy,
        device=DEVICE, freeze=True, hf_cache_dir=HF_CACHE_DIR,
    )

    _smoke_embeddings = extract_embeddings(
        _smoke_encoder, _smoke_tokenizer, smoke_df, base_config, DEVICE,
        cache_path=None,
    )
    print("Smoke embeddings shape:", _smoke_embeddings.shape)
    assert _smoke_embeddings.shape[0] == len(smoke_df)
    import numpy as np
    assert np.isfinite(_smoke_embeddings).all(), "Non-finite values slipped past the guardrail -- investigate before proceeding."

    print("Smoke test passed: encoder loads, tokenizes, extracts finite embeddings.")

    del _smoke_encoder, _smoke_tokenizer, _smoke_embeddings
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")


## 11. Extract frozen NeoBERT embeddings for the full dataset

In [ ]:
try:
    print("Loading tokenizer...")
    tokenizer = load_code_tokenizer(base_config.tokenizer_name, hf_cache_dir=HF_CACHE_DIR)
    print("Tokenizer loaded")

    print("Loading encoder...")
    encoder = load_code_encoder(
        base_config.model_name,
        dtype_policy=base_config.dtype_policy,
        device=DEVICE,
        freeze=True,
        hf_cache_dir=HF_CACHE_DIR,
    )
    print("Encoder loaded successfully on", DEVICE)

    print("\nExtracting embeddings for the full dataset...")
    dataset_embeddings = extract_embeddings(
        encoder, tokenizer, dataset_frame, base_config, DEVICE,
        cache_path=EMBEDDING_CACHE_DIR / "dataset_embeddings.npy",
    )
    print("Embeddings extracted:", dataset_embeddings.shape)

    del encoder
    torch.cuda.empty_cache()
    print("VRAM cleaned up")

except RuntimeError as e:
    if "NaN/Inf detected" in str(e):
        print("\n" + "="*70)
        print("ERROR: NeoBERT numerical-instability bug triggered")
        print("="*70)
        print(str(e))
        print("\nDo not silently drop/zero these rows. Investigate dtype_policy "
              "(try dtype_policy='float32' on base_config) before re-running.")
    raise
except Exception as e:
    print(f"ERROR during embedding extraction: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    raise


## 12. Single-fold inner profile

In [ ]:
if RUN_PROFILE:
    profile = run_exp3_nested_inner_profile(
        dataset_frame=dataset_frame,
        dataset_embeddings=dataset_embeddings,
        manifest=manifest_df,
        outer_fold_id=4,
        base_config=base_config,
        nested_config=nested_config,
    )
    print("Profile duration minutes:", profile["total_profile_seconds"] / 60)
    display(pd.DataFrame([profile["selected"]]))
    display(profile["C_summary"])
else:
    print("RUN_PROFILE=False; skipping.")


## 13. Official rotating 5-fold run

In [ ]:
if RUN_OFFICIAL:
    results = run_exp3_nested_probe(
        dataset_frame=dataset_frame,
        dataset_embeddings=dataset_embeddings,
        manifest=manifest_df,
        base_config=base_config,
        nested_config=nested_config,
        output_dir=EXP3_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(DOWNSAMPLED_PARQUET),
            "manifest_path": str(MANIFEST_PATH),
        },
    )
    print("Pooled OOF metrics (secondary cross-check):")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in results["evaluation"]["pooled_metrics"].items()]))
    print("Mean +/- std across the 5 outer folds (headline result):")
    display(results["evaluation"]["fold_summary"])
    print("Selected hyperparameters by outer fold:")
    display(results["selected"])
else:
    results = None
    print("RUN_OFFICIAL=False; skipping.")


## 14. Confidence interval on pooled OOF PR-AUC (ad hoc)

In [ ]:
exp3_oof_ci = bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(format_ci_report(exp3_oof_ci))


## 15. Cleanup

In [26]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP3_OUTPUT_DIR}.")


VRAM allocated: 0.00851968 GB
Disk usage at /workspace: 5235.7 GB used / 5714.2 GB total (190.4 GB free)
